In [1]:
print('Bismillahir Rahmanir Rahiym!')
#FOR LEARNING ONLY

import numpy as np 
import pandas as pd
import seaborn as sns
sns.set(color_codes=True)
import matplotlib.pyplot as plt
%matplotlib inline

import hyperopt
from catboost import Pool, CatBoostClassifier, cv
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

Bismillahir Rahmanir Rahiym!


In [2]:
train_df=pd.read_csv("../input/titanic/train.csv")
test_df=pd.read_csv("../input/titanic/test.csv")

In [3]:
train_df.shape, test_df.shape

((891, 12), (418, 11))

In [4]:
#show how many the null value for each column
train_df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [5]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [6]:
#for the train data ,the age ,fare and embarked has null value,so just make it -999 for it
#and the catboost will distinguish it
train_df.fillna(-999,inplace=True)
test_df.fillna(-999,inplace=True)

In [7]:
#now we will get the train data and label
x = train_df.drop('Survived',axis=1)
y = train_df.Survived

In [8]:
#show what the dtype of x, note that the catboost will just make the string object to categorical 
#object inside
x.dtypes

PassengerId      int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object
dtype: object

In [9]:
#choose the features we want to train, just forget the float data
cate_features_index = np.where(x.dtypes != float)[0]

In [10]:
#make the x for train and test (also called validation data) 
xtrain,xtest,ytrain,ytest = train_test_split(x,y,train_size=.85,random_state=1234)

In [11]:
#let us make the catboost model, use_best_model params will make the model prevent overfitting
model = CatBoostClassifier(eval_metric='Accuracy',
                           use_best_model=True,
                           random_seed=42)

In [12]:
#now just to make the model to fit the data
model.fit(xtrain,ytrain,cat_features=cate_features_index,
          eval_set=(xtest,ytest))

Learning rate set to 0.029583
0:	learn: 0.8124174	test: 0.8059701	best: 0.8059701 (0)	total: 59.7ms	remaining: 59.6s
1:	learn: 0.8110964	test: 0.8059701	best: 0.8059701 (0)	total: 63.1ms	remaining: 31.5s
2:	learn: 0.8163804	test: 0.7985075	best: 0.8059701 (0)	total: 70.1ms	remaining: 23.3s
3:	learn: 0.8150594	test: 0.7985075	best: 0.8059701 (0)	total: 75.9ms	remaining: 18.9s
4:	learn: 0.8190225	test: 0.7985075	best: 0.8059701 (0)	total: 80.8ms	remaining: 16.1s
5:	learn: 0.8269485	test: 0.8059701	best: 0.8059701 (0)	total: 86.9ms	remaining: 14.4s
6:	learn: 0.8269485	test: 0.8059701	best: 0.8059701 (0)	total: 96.4ms	remaining: 13.7s
7:	learn: 0.8216645	test: 0.8059701	best: 0.8059701 (0)	total: 104ms	remaining: 12.9s
8:	learn: 0.8216645	test: 0.8059701	best: 0.8059701 (0)	total: 107ms	remaining: 11.7s
9:	learn: 0.8256275	test: 0.7985075	best: 0.8059701 (0)	total: 113ms	remaining: 11.2s
10:	learn: 0.8216645	test: 0.8134328	best: 0.8134328 (10)	total: 118ms	remaining: 10.6s
11:	learn: 0.82

In [13]:

cv_data = cv(Pool(xtrain,ytrain,cat_features=cate_features_index),model.get_params(),fold_count=10)
cv_params = model.get_params()
cv_params.update({
    'logging_level': None,
})

CatBoostError: Parameter loss_function should be specified for cross-validation

In [14]:
#last let us make the submission,note that you have to make the pred to be int!
pred = model.predict(test_df)
pred = pred.astype(np.int)
submission = pd.DataFrame({'PassengerId':test_df['PassengerId'],'Survived':pred})

In [15]:
#make the file to yourself's directory
submission.to_csv('catboost.csv',index=False)